# Этап 3: IrResnet4 multi-label (автономный)

3-канальный вход 400–4000 см⁻¹ + контекст ATR/gas/solution.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Lamblador/IR_expert_system_3.git"
REPO_BRANCH = "colab-v1"
REPO_DIR_NAME = "IR_expert_system_3"

def _run_git(cmd: list[str], cwd: Path | None = None) -> None:
    print('git', ' '.join(cmd), f'(cwd={cwd})' if cwd else '')
    subprocess.run(cmd, cwd=cwd, check=True)

def _ensure_repo_at(repo_dir: Path) -> None:
    if (repo_dir / '.git').is_dir():
        _run_git(['git', 'fetch', 'origin', REPO_BRANCH], cwd=repo_dir)
        _run_git(['git', 'checkout', REPO_BRANCH], cwd=repo_dir)
        _run_git(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=repo_dir)
    else:
        if repo_dir.exists():
            raise RuntimeError(f'{repo_dir} существует, но это не git-репозиторий')
        _run_git([
            'git', 'clone', '-b', REPO_BRANCH, '--single-branch',
            REPO_URL, str(repo_dir),
        ])
    rev = subprocess.check_output(
        ['git', 'rev-parse', '--short', 'HEAD'], cwd=repo_dir, text=True
    ).strip()
    print(f'Репозиторий: {repo_dir.resolve()} @ {REPO_BRANCH} ({rev})')

cwd = Path.cwd()
in_colab = Path('/content').exists() and str(cwd).startswith('/content')
local_repo = (cwd / 'pyproject.toml').is_file()

if local_repo and not in_colab:
    ROOT = cwd.resolve()
    print('Локальный репозиторий (ветку не переключаем):', ROOT)
elif (cwd / REPO_DIR_NAME / 'pyproject.toml').is_file():
    ROOT = (cwd / REPO_DIR_NAME).resolve()
    _ensure_repo_at(ROOT)
elif Path(f'/content/{REPO_DIR_NAME}/pyproject.toml').is_file():
    ROOT = Path(f'/content/{REPO_DIR_NAME}').resolve()
    _ensure_repo_at(ROOT)
else:
    ROOT = (Path('/content') / REPO_DIR_NAME if in_colab else cwd / REPO_DIR_NAME).resolve()
    _ensure_repo_at(ROOT)

import os
os.chdir(ROOT)
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('cd', str(ROOT))
except Exception:
    pass
print('ROOT:', ROOT.resolve())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[torch]'], check=True)
help_txt = subprocess.check_output(['ir-pipeline', '--help'], text=True)
if ' run ' not in help_txt:
    print('WARNING: команда `run` отсутствует. Ноутбук использует fallback без run-stage.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
IR_DATA = Path('/content/drive/MyDrive/ir_data')
print('IR_DATA exists:', IR_DATA.exists(), IR_DATA)


## Датасет: загрузка вручную

1. **Files → Upload** в Colab: `dataset_v001.zip` / `dataset_mini.zip` в `/content` (или положите архив на Google Drive).
2. Выполните ячейку распаковки ниже — ожидается `data/processed/<версия>/spectra.npz`.
3. Если архива нет — следующая ячейка скачает мини-датасет с Hugging Face.


In [ ]:
from pathlib import Path
import zipfile

DATASET_VERSIONS = ('dataset_mini', 'dataset_v001')
SEARCH_ROOTS = [
    Path('/content'),
    Path('/content/IR_expert_system_3'),
    Path('/content/drive/MyDrive'),
    Path('/content/drive/MyDrive/ir_data'),
    Path('.'),
]
try:
    SEARCH_ROOTS.insert(0, IR_DATA)
except NameError:
    pass
DEST = Path('data/processed')
DEST.mkdir(parents=True, exist_ok=True)

def _dataset_ready(name: str) -> bool:
    return (DEST / name / 'spectra.npz').is_file()

def _find_zip_archives() -> list[Path]:
    found: list[Path] = []
    seen: set[str] = set()
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob('*.zip'):
            key = str(p.resolve())
            if key in seen:
                continue
            low = p.name.lower()
            if any(v in low for v in DATASET_VERSIONS):
                seen.add(key)
                found.append(p)
    return sorted(found, key=lambda x: x.stat().st_mtime, reverse=True)

archives = _find_zip_archives()
print('Найденные zip с датасетом:')
if archives:
    for p in archives[:15]:
        print(f'  {p} ({p.stat().st_size / 1e6:.1f} MB)')
else:
    print('  (нет — загрузите через Files → Upload)')

for version in DATASET_VERSIONS:
    if _dataset_ready(version):
        print(f'OK: {DEST / version} уже распакован')
        continue
    matched = [p for p in archives if version in p.name.lower()]
    if not matched:
        print(f'Пропуск {version}: zip не найден')
        continue
    zp = matched[0]
    print(f'Распаковка {zp.name} → {DEST}')
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(DEST)
    if _dataset_ready(version):
        print(f'  → готово: {DEST / version / "spectra.npz"}')
    else:
        print(
            f'  WARNING: после распаковки нет {DEST / version / "spectra.npz"}. '
            'Проверьте структуру zip (внутри должна быть папка {version}/).'
        )


In [ ]:
from pathlib import Path

DATASET_DIR = Path('data/processed/dataset_mini')
if DATASET_DIR.joinpath('spectra.npz').is_file():
    print(f'OK: {DATASET_DIR}')
else:
    print('dataset_mini not found → fetching from HF...')
    !ir-pipeline fetch-data --filename dataset_mini.zip --extract-to data/processed


## Гиперпараметры обучения (CNN / IrResnet)

| Параметр | По умолчанию (Colab) | Файл / как поменять |
|----------|----------------------|---------------------|
| **Эпохи** | `torch_epochs: 30` | `configs/train_irresnet_colab.yaml` |
| **Learning rate** | `torch_lr: 0.001` | тот же yaml |
| **Batch size** | `torch_batch_size: 32` | тот же yaml |
| **Оптимизатор** | `torch_optimizer: adamw` | `adamw` \| `adam` \| `sgd` |
| **Метки** | `structure` / `structure_smarts` / `spectrum` | `--label-schema` в CLI или kwarg в `train_irresnet_run` |
| **Loss (IrResnet)** | `torch_loss: bce_with_logits` | multi-label BCE с logits |
| **Loss (torch-train 1D CNN)** | `smooth_l1` | в `configs/train_torch_colab.yaml`: `smooth_l1` или `mse` |
| **Размер скрытого слоя** | `ir_hidden_size: 34` | только IrResnet |
| **Live-графики** | `live_training_plot: true` | в Colab: clear + график каждую эпоху; лог — последние 5 значений |

В ячейке обучения ниже используется `--config configs/train_irresnet_colab.yaml`. Скопируйте yaml, измените числа, сохраните и укажите свой путь в `--config`.


In [ ]:
%matplotlib inline
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, merge_train_defaults, resolve_paths
from ir_pipeline.irresnet_train import train_irresnet_run

paths = resolve_paths(load_yaml(Path('configs/paths.huggingface.yaml')))
from ir_pipeline.config_loader import resolve_dataset_dir
paths_cfg = load_yaml(Path('configs/paths.huggingface.yaml'))
DATASET = resolve_dataset_dir(paths_cfg, 'auto')
cfg_path = Path('configs/train_irresnet_original.yaml') if 'v003' in str(DATASET) else Path('configs/train_irresnet_colab.yaml')
train_cfg = merge_train_defaults(load_yaml(cfg_path))
RUN_DIR = Path('runs/colab_pipeline_irresnet/irresnet_run')
summary = train_irresnet_run(
    dataset_dir=DATASET,
    run_dir=RUN_DIR,
    bands_yaml=paths['bands_config'],
    train_cfg=train_cfg,
    label_schema=train_cfg.get('label_schema', 'structure_smarts'),
)
print(summary)


## Сохранить обученную модель на локальный ПК

Выполните ячейку ниже — браузер скачает zip каталога run (`models.joblib`, `metrics.json`, `irresnet_bundle.pt` и т.д.). На Windows распакуйте в `runs/<имя>/` и укажите `--run-dir`.


In [ ]:
from pathlib import Path
import shutil
from google.colab import files

RUN_DIR = Path('runs/colab_pipeline_irresnet/irresnet_run')
if not RUN_DIR.is_dir():
    raise FileNotFoundError(
        f'Нет {RUN_DIR} — сначала выполните ячейку обучения.'
    )

artifacts = [p for p in RUN_DIR.iterdir() if p.is_file()]
if not artifacts:
    raise FileNotFoundError(f'{RUN_DIR} пуст — нечего архивировать.')
print('Файлы:', [p.name for p in sorted(artifacts)])

zip_path = Path('/content/irresnet_run_colab.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', RUN_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f'Архив: {zip_path} ({size_mb:.2f} MB)')
files.download(str(zip_path))
print('Скачивание запущено.')
